# Subsecuencia común más larga (LCS)

Una subsecuencia se obtiene borrando elementos sin cambiar el orden de los que quedan. $L[i][j]$ es la longitud de la LCS de los prefijos $a_{1..i}$ y $b_{1..j}$: $L[i][j] = L[i-1][j-1] + 1$ si $a_i = b_j$ y, si no, $\max(L[i-1][j],\ L[i][j-1])$, con $L[i][0] = L[0][j] = 0$. Cuesta $O(n \cdot m)$.

## 1. Tabla de longitudes

Calcular y mostrar la tabla para `ABCBDAB` y `BDCABA`, y la longitud de la LCS de dos cadenas de ADN.

In [1]:
def tabla_lcs(a, b):
    L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                L[i][j] = L[i - 1][j - 1] + 1
            else:
                L[i][j] = max(L[i - 1][j], L[i][j - 1])
    return L

a, b = "ABCBDAB", "BDCABA"
L = tabla_lcs(a, b)
print("    " + " ".join(f"{x:>2}" for x in " " + b))
for letra, fila in zip(" " + a, L):
    print(f"{letra:>2}  " + " ".join(f"{x:>2}" for x in fila))

adn1 = "ACCGGTCGAGTGCGCGGAAGCCGGCCGAA"
adn2 = "GTCGTTCGGAATGCCGTTGCTCTGTAAA"
print(tabla_lcs(adn1, adn2)[-1][-1])

        B  D  C  A  B  A
     0  0  0  0  0  0  0
 A   0  0  0  0  1  1  1
 B   0  1  1  1  1  2  2
 C   0  1  1  2  2  2  2
 B   0  1  1  2  2  3  3
 D   0  1  2  2  2  3  3
 A   0  1  2  2  3  3  4
 B   0  1  2  2  3  4  4
20


## 2. Reconstrucción

Recorrer la tabla desde la esquina: si los elementos coinciden forman parte de la LCS; si no, ir hacia la casilla vecina con mayor valor. Comprobar contra fuerza bruta (probar las subsecuencias de una cadena de mayor a menor longitud) con cadenas aleatorias.

In [2]:
import random
from itertools import combinations

def lcs(a, b):
    L = tabla_lcs(a, b)
    i, j, resultado = len(a), len(b), []
    while i and j:
        if a[i - 1] == b[j - 1]:
            resultado.append(a[i - 1])
            i, j = i - 1, j - 1
        elif L[i - 1][j] >= L[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return resultado[::-1]

def es_subsecuencia(s, t):
    resto = iter(t)
    return all(x in resto for x in s)

def lcs_fuerza_bruta(a, b):
    for r in range(len(a), -1, -1):
        if any(es_subsecuencia(c, b) for c in combinations(a, r)):
            return r

print("".join(lcs(a, b)), "".join(lcs(adn1, adn2)))

random.seed(5)
for _ in range(300):
    x = "".join(random.choices("ACGT", k=random.randint(0, 9)))
    y = "".join(random.choices("ACGT", k=random.randint(0, 9)))
    s = lcs(x, y)
    assert len(s) == lcs_fuerza_bruta(x, y) and es_subsecuencia(s, x) and es_subsecuencia(s, y)

BCBA GTCGTCGGAAGCCGGCCGAA


## 3. Versión con memoización

Escribir la recurrencia directamente como función recursiva con `functools.cache`, comprobar que coincide con la tabla y contar cuántos subproblemas distintos resuelve. La recursión llega a profundidad $n + m$, así que no sirve para textos largos sin subir el límite de recursión.

In [3]:
from functools import cache

def lcs_memo(a, b):
    @cache
    def L(i, j):
        if i == 0 or j == 0:
            return 0
        if a[i - 1] == b[j - 1]:
            return L(i - 1, j - 1) + 1
        return max(L(i - 1, j), L(i, j - 1))

    return L(len(a), len(b)), L.cache_info().currsize

print(lcs_memo(a, b))
longitud, subproblemas = lcs_memo(adn1, adn2)
print(f"ADN: longitud {longitud}, {subproblemas} subproblemas de {(len(adn1) + 1) * (len(adn2) + 1)} casillas")

random.seed(6)
for _ in range(300):
    x = "".join(random.choices("ab", k=random.randint(0, 30)))
    y = "".join(random.choices("ab", k=random.randint(0, 30)))
    assert lcs_memo(x, y)[0] == tabla_lcs(x, y)[-1][-1]

(4, 30)
ADN: longitud 20, 481 subproblemas de 870 casillas


## 4. Diferencias entre dos textos

`diff` compara dos versiones de un texto con la LCS de sus **líneas**: las comunes se mantienen, las que solo están en la primera se borran (`-`) y las que solo están en la segunda se añaden (`+`). Comprobar que a partir del resultado se recuperan las dos versiones.

In [4]:
def diff(antes, despues):
    L = tabla_lcs(antes, despues)
    i, j, salida = len(antes), len(despues), []
    while i or j:
        if i and j and antes[i - 1] == despues[j - 1]:
            salida.append("  " + antes[i - 1])
            i, j = i - 1, j - 1
        elif j and (not i or L[i][j - 1] >= L[i - 1][j]):
            salida.append("+ " + despues[j - 1])
            j -= 1
        else:
            salida.append("- " + antes[i - 1])
            i -= 1
    return salida[::-1]

antes = [
    "Papas arrugadas",
    "1 kg de papas pequeñas",
    "sal gorda",
    "agua",
    "Cocer las papas con mucha sal.",
    "Escurrir y dejar secar al fuego.",
    "Servir con mojo rojo.",
]
despues = [
    "Papas arrugadas",
    "1 kg de papas negras pequeñas",
    "sal gorda",
    "agua",
    "Cocer las papas con mucha sal durante 20 minutos.",
    "Escurrir y dejar secar al fuego.",
    "Servir con mojo rojo y mojo verde.",
    "Acompañar con queso asado.",
]
cambios = diff(antes, despues)
print("\n".join(cambios))

def versiones(cambios):
    return [l[2:] for l in cambios if l[0] in " -"], [l[2:] for l in cambios if l[0] in " +"]

assert versiones(cambios) == (antes, despues)
random.seed(7)
for _ in range(300):
    x = random.choices(["a", "b", "c", "d"], k=random.randint(0, 10))
    y = random.choices(["a", "b", "c", "d"], k=random.randint(0, 10))
    cambios = diff(x, y)
    assert versiones(cambios) == (x, y) and sum(l[0] == " " for l in cambios) == len(lcs(x, y))

  Papas arrugadas
- 1 kg de papas pequeñas
+ 1 kg de papas negras pequeñas
  sal gorda
  agua
- Cocer las papas con mucha sal.
+ Cocer las papas con mucha sal durante 20 minutos.
  Escurrir y dejar secar al fuego.
- Servir con mojo rojo.
+ Servir con mojo rojo y mojo verde.
+ Acompañar con queso asado.
